# Regressor only tasks

## Imports and constants

In [1]:
import dsi #De Simone Inference module
import itertools
import time
import importlib
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml

RANDOM_STATE = int('8'*8)
DARK_MODE = True
VAL_SIZE = 0.15
TEST_SIZE = 15/85

## Routines functions

In [3]:
def load_dataset_and_split(
    uci_id,
    shuffle = True,
    ):
    """
    Automatically load an UCI dataset from ID (OpenML)
    Gives train, validation and test dataframes in xp backend.
    """
    
    # Loading data
    data = fetch_openml(data_id=uci_id, as_frame=False)

    X = data.data
    y = data.target

    # Building dataframe
    df = np.column_stack([X, y])

    # Test split
    df_train, df_test = train_test_split(
        df,
        test_size=TEST_SIZE,
        shuffle=shuffle,
        random_state=RANDOM_STATE,
    )

    # Validation split
    df_train, df_val = train_test_split(
        df_train,
        test_size=VAL_SIZE,
        shuffle=shuffle,
        random_state=RANDOM_STATE,
    )

    return df_train, df_val, df_test


def generate_configurations(unique_list):
    """
    Generate all 8 combinations of DSI flags.

    parameters
    ----------
    unique_list: list
        List of bool: True where operator forces the division of feature domain
        on the unique values contained. False where the approach follows density
        based or linear feature domain division.

    returns
    -------
    configs: dict
        Various configuration to test the model.
    """

    configs = []

    fire_options = [True, False]
    fuzzy_options = [True, False]
    db_options = [True, False]

    for fire, fuzzy, db in itertools.product(fire_options, fuzzy_options, db_options):

        # Base division depending on DB flag
        if db:
            division = ["D"] * len(unique_list)
        else:
            division = ["L"] * len(unique_list)

        # Enforce class-based division
        for i, is_unique in enumerate(unique_list):
            if is_unique:
                division[i] = "U"

        configs.append({
            "fire_of_train_rules": fire,
            "fuzzy_output": fuzzy,
            "default_division": division,
        })

    return configs


def regression_test(df_train, df_val, df_test, configurations):
    """
    Test all 8 configuration for regression task solved by De Simone Inference
    """
    results = []
    
    for config in configurations:

        print("Starting optimizzation for:\n",config)
        
        model = dsi.DSregressor(**config)
        model.fit_cross_validation(df_train, df_val)

        # --- Predizione ---
        y_out = model.predict(df_test[:, :-1])
        y_true = df_test[:, -1]
        y_pred = y_out.get()

        # --- Gestione predizioni mancanti ---
        valid_mask = ~np.isnan(y_pred)
        cov = valid_mask.mean()   # coverage

        if valid_mask.sum() == 0:
            y_pred_valid = np.array([])
            y_true_valid = np.array([])
        else:
            y_pred_valid = y_pred[valid_mask]
            y_true_valid = y_true[valid_mask]

        # --- Entry base ---
        entry = {
            "fire_of_train_rules": config["fire_of_train_rules"],
            "fuzzy_output": config["fuzzy_output"],
            "default_division": config["default_division"],
            "cov": cov
        }

        # --- Score ---
        if len(y_pred_valid) > 0:
            entry["R2"] = dsi.r2(y_true_valid, y_pred_valid).get()
            entry["MAE"] = dsi.mae(y_true_valid, y_pred_valid).get()
            entry["RMSE"] = dsi.rmse(y_true_valid, y_pred_valid).get()
        else:
            entry["r2"] = np.nan
            entry["MAE"] = np.nan
            entry["RMSE"] = np.nan

        results.append(entry)

    return results


def collapse_rare_categories(df, max_levels=1000):
    df = df.copy()
    cat_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in cat_cols:
        counts = df[col].value_counts()

        # Se la cardinalità è già sotto la soglia, non fare nulla
        if len(counts) <= max_levels:
            continue

        # Prendi le categorie più frequenti
        keep = counts.nlargest(max_levels).index

        # Collassa le categorie rare
        df[col] = df[col].where(df[col].isin(keep), "__OTHER__")

    return df


def one_hot_encode(df):
    df = df.copy()
    cat_cols = df.select_dtypes(include=["object", "category"]).columns

    if len(cat_cols) == 0:
        return df  # niente da codificare

    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    encoded = encoder.fit_transform(df[cat_cols])

    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols))

    # Rimuovi le colonne categoriche originali
    df = df.drop(columns=cat_cols)

    # Aggiungi le colonne one-hot
    df = pd.concat([df.reset_index(drop=True), encoded_df.reset_index(drop=True)], axis=1)

    return df


# Pipeline completa
def preprocess_ctr23_style(df, max_levels=1000):
    df = collapse_rare_categories(df, max_levels=max_levels)
    df = one_hot_encode(df)
    return df


## Test on datasets